# Comparaison pipelines ML — features et options

On compare :
- **Features :** simple, hog, lbp, haralick, combined
- **Modèles :** Random Forest, régression logistique, XGBoost (si dispo)
- **Options :** avec/sans SMOTE, seuil fixe 0.5 vs seuil optimisé pour 95% spécificité

Tout en ML classique (pas de deep learning).

In [4]:
%pip install scikit-image imbalanced-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Cellule 1 — instantanée (chemin + données uniquement)
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "train_labels.csv").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loading import get_ml_subset_df, get_train_val_test_splits
print("Chemin projet OK, data_loading chargé.")

Chemin projet OK, data_loading chargé.


**Imports lourds** (numpy, pandas, sklearn, features, pipeline) — peut prendre **1 à 2 min** la première fois. À lancer une seule fois.

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from src.feature_extraction import build_feature_matrix
from src.evaluation_metrics import roc_auc, sensitivity_at_specificity, partial_auc, print_metrics
from src.pipeline_ml import get_smote, get_classifier, threshold_predict, find_threshold_for_specificity
print("Tous les imports sont prêts.")

Tous les imports sont prêts.


## 1. Données (échantillon pour tests rapides)

In [7]:
root = PROJECT_ROOT
df = get_ml_subset_df()
train_df, val_df, test_df = get_train_val_test_splits(df)
MAX_TRAIN, MAX_VAL = 3000, 800

X_train, y_train = build_feature_matrix(train_df, root=root, feature_type="simple", max_samples=MAX_TRAIN)
X_val, y_val = build_feature_matrix(val_df, root=root, feature_type="simple", max_samples=MAX_VAL)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
print("Train:", X_train_s.shape, "Val:", X_val_s.shape)

Features (simple): 100%|██████████| 800/800 [00:17<00:00, 46.90it/s]

Train: (3000, 102) Val: (800, 102)


## 2. Test SMOTE + seuil optimisé (features simple, RF)

In [8]:
X_tr, y_tr = get_smote(X_train_s, y_train)
print("Après SMOTE:", X_tr.shape, "— RG:", y_tr.sum())

clf = get_classifier("rf")
clf.fit(X_tr, y_tr)
y_val_score = clf.predict_proba(X_val_s)[:, 1]

thresh = find_threshold_for_specificity(y_val, y_val_score, target_spec=0.95)
y_val_pred_thresh = threshold_predict(clf.predict_proba(X_val_s), thresh)
print("Seuil optimisé (95% spec):", round(thresh, 4))
print_metrics(y_val, y_val_pred_thresh, y_val_score)

Après SMOTE: (5800, 102) — RG: 2900
Seuil optimisé (95% spec): inf
[[776   0]
 [ 24   0]]
AUC-ROC:                    0.6131
pAUC (90-100% spec):        0.4848
Sensibilité @ 95% spec:     0.0000
Sensibilité @ 90% spec:     0.0833


## 3. Comparaison rapide : type de features (RF, sans SMOTE, seuil 0.5)

In [9]:
feature_types = ["simple", "hog", "lbp", "haralick", "combined"]
results = []
for ft in feature_types:
    try:
        Xt, yt = build_feature_matrix(train_df, root=root, feature_type=ft, max_samples=MAX_TRAIN)
        Xv, yv = build_feature_matrix(val_df, root=root, feature_type=ft, max_samples=MAX_VAL)
        sc = StandardScaler().fit(Xt)
        Xt_s, Xv_s = sc.transform(Xt), sc.transform(Xv)
        clf = get_classifier("rf")
        clf.fit(Xt_s, yt)
        score = clf.predict_proba(Xv_s)[:, 1]
        auc = roc_auc(yv, score)
        se95 = sensitivity_at_specificity(yv, score, 0.95)
        results.append({"features": ft, "AUC": auc, "Sens@95": se95})
        print(ft, "— AUC:", round(auc, 4), "Sens@95:", round(se95, 4))
    except Exception as e:
        print(ft, "— Erreur:", e)
        results.append({"features": ft, "AUC": None, "Sens@95": None})
pd.DataFrame(results)

Features (simple):   0%|          | 0/3000 [00:00<?, ?it/s]

Features (simple): 100%|██████████| 800/800 [00:17<00:00, 44.50it/s]


simple — AUC: 0.5832 Sens@95: 0.0417


Features (hog): 100%|██████████| 800/800 [00:17<00:00, 44.67it/s]


hog — AUC: 0.4955 Sens@95: 0.125


...

Features (haralick): 100%|██████████| 800/800 [00:30<00:00, 25.98it/s]


haralick — AUC: 0.6006 Sens@95: 0.2083


Features (combined): 100%|██████████| 800/800 [00:41<00:00, 19.38it/s]


combined — AUC: 0.6925 Sens@95: 0.2083


,features,AUC,Sens@95
0,simple,0.583172,0.041667
1,hog,0.495517,0.125000
2,lbp,0.625188,0.125000
3,haralick,0.600650,0.208333
4,combined,0.692520,0.208333


## 4. Comparaison modèles (features = combined si dispo, sinon simple)

In [ ]:
ft = "combined"  # ou "simple" si combined trop lent
Xt, yt = build_feature_matrix(train_df, root=root, feature_type=ft, max_samples=MAX_TRAIN)
Xv, yv = build_feature_matrix(val_df, root=root, feature_type=ft, max_samples=MAX_VAL)
Xt, yt = get_smote(Xt, yt)
sc = StandardScaler().fit(Xt)
Xt_s, Xv_s = sc.transform(Xt), sc.transform(Xv)

for name in ["rf", "lr", "xgb"]:
    try:
        clf = get_classifier(name)
        if name == "xgb":
            n_neg, n_pos = int((yt == 0).sum()), int((yt == 1).sum())
            clf.set_params(scale_pos_weight=n_neg / max(1, n_pos))
        clf.fit(Xt_s, yt)
        score = clf.predict_proba(Xv_s)[:, 1]
        print(name, "— AUC:", round(roc_auc(yv, score), 4), "Sens@95:", round(sensitivity_at_specificity(yv, score, 0.95), 4))
    except Exception as e:
        print(name, "—", e)

Features (combined): 100%|██████████| 800/800 [00:35<00:00, 22.78it/s]


rf — AUC: 0.6108 Sens@95: 0.0833
